# Shot-location pipeline

Thin notebook over `src.pipeline`. One pass does **SAM2 player tracking**, **team clustering**, and **shot locations** (court x/y, made/miss).

Change the **Config** cell, then re-run the launch cell. Keep the pipeline object to avoid reloading models.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
print(REPO_ROOT)

C:\Users\zecab\.gemini\antigravity-ide\scratch\Basketball-player-tracking


In [2]:
import os

os.environ["CORE_MODEL_SAM_ENABLED"] = "False"
os.environ["CORE_MODEL_SAM3_ENABLED"] = "False"
os.environ["CORE_MODEL_YOLO_WORLD_ENABLED"] = "False"
os.environ["CORE_MODEL_GAZE_ENABLED"] = "False"
os.environ.setdefault(
    "ONNXRUNTIME_EXECUTION_PROVIDERS",
    "CUDAExecutionProvider,CPUExecutionProvider",
)

from src.utils.env import load_api_keys

load_api_keys()

In [3]:
import cv2
import pandas as pd
import supervision as sv

from src.pipeline import (
    DEFAULT_TEAM_NAMES,
    TEAM_COLORS,
    TEAM_ROSTERS,
    BasketballPipeline,
    plot_shot_chart,
    run_shot_location_pipeline,
)

[09/01/26 13:03:16] WARNING  Your inference package version 1.5.0 is out of date! Please upgrade to  ]8;id=5914677;file://c:\Users\zecab\.gemini\antigravity-ide\scratch\Basketball-player-tracking\.venv\Lib\site-packages\inference\core\__init__.py\__init__.py]8;;\:]8;id=5914678;file://c:\Users\zecab\.gemini\antigravity-ide\scratch\Basketball-player-tracking\.venv\Lib\site-packages\inference\core\__init__.py#50\50]8;;\
                             version 1.5.1 of inference for the latest features and bug fixes by                   
                             running `pip install --upgrade inference`.                                            

[transformers] `DepthProImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `DepthProImageProcessor` instead.


## Config

Edit this cell, then re-run **Launch**. `None` for `MAX_FRAMES` processes the whole clip. `TARGET_FPS = None` uses every source frame.
Swap `TEAM_NAMES` 0/1 if jersey colors landed on the wrong team.

In [4]:
VIDEO_NAME = "Bad_detections_game2_1min.mp4"
VIDEO = DATA_DIR / VIDEO_NAME
if not VIDEO.exists():
    clips = sorted([*DATA_DIR.glob("*.mp4"), *DATA_DIR.glob("*.mov")])
    if not clips:
        raise FileNotFoundError(
            f"No video at {VIDEO}. Put a clip in data/ or change VIDEO_NAME."
        )
    VIDEO = clips[0]

MAX_FRAMES = 200          # None = full clip
TARGET_FPS = 10          # None = every frame
USE_OCR = False
SAVE_HISTORY = False     # JPEGs + SAM2 dumps under outputs/<clip>/history

TEAM_NAMES = dict(DEFAULT_TEAM_NAMES)
# TEAM_NAMES = {0: "New York Knicks", 1: "Boston Celtics"}

print(VIDEO)
print("teams:", TEAM_NAMES)

C:\Users\zecab\.gemini\antigravity-ide\scratch\Basketball-player-tracking\data\Bad_detections_game2_1min.mp4
teams: {0: 'Boston Celtics', 1: 'New York Knicks'}


## Launch

First cell loads RF-DETR, SAM2, and the team classifier (slow). Second cell runs tracking + shot locations. Re-run only the second cell after changing Config.

In [5]:
pipeline = BasketballPipeline(
    team_names=TEAM_NAMES,
    team_rosters=TEAM_ROSTERS,
    use_ocr=USE_OCR,
)

FileNotFoundError: SAM2 repo not found at C:\Users\zecab\.gemini\antigravity-ide\scratch\segment-anything-2-real-time. Clone segment-anything-2-real-time there or set SAM2_DIR.

In [ ]:
run = run_shot_location_pipeline(
    VIDEO,
    output_dir=OUTPUT_DIR,
    max_frames=MAX_FRAMES,
    target_fps=TARGET_FPS,
    team_names=TEAM_NAMES,
    use_ocr=USE_OCR,
    save_history=SAVE_HISTORY,
    plot=True,
    pipeline=pipeline,
)

identity_df = run.result.identity_df
player_df = run.result.player_df
event_df = run.result.event_df
shots_df = run.result.shots_df

## Results

`identity_df` is one row per SAM2 track (team + optional jersey). `player_df` is court position every processed frame. `shots_df` is the shot chart table.

In [ ]:
identity_df

In [ ]:
player_df.head(20)

In [ ]:
shots_df

In [ ]:
if run.chart_path is not None and run.chart_path.exists():
    sv.plot_image(cv2.imread(str(run.chart_path)))
else:
    print("No shot chart (no shots or plot=False).")

Optional: chart one tracker or one team without re-running detection.

In [ ]:
TRACKER_ID = None          # e.g. 3
TEAM = None                # e.g. "Boston Celtics"

chart_name = "shot_chart"
if TRACKER_ID is not None:
    chart_name += f"_tracker{TRACKER_ID}"
if TEAM:
    chart_name += "_" + TEAM.replace(" ", "_").lower()
filtered_chart = run.run_dir / f"{chart_name}.jpg"

plot_shot_chart(
    shots_df,
    filtered_chart,
    tracker_id=TRACKER_ID,
    team=TEAM,
)
sv.plot_image(cv2.imread(str(filtered_chart)))